# OData Filter Generation Experiments with OpenAI

This notebook helps you convert a natural-language question into the **filter part** of an OData query.

Design goals:
- Inject field metadata and instructions so the model maps business terms to the correct fields.
- Add few-shot examples for better accuracy.
- Return only the OData filter expression (safe to append to `$filter=` in a URL).


In [ ]:
import os
from textwrap import dedent
from openai import OpenAI


In [ ]:
# Set your API key before running this notebook:
# export OPENAI_API_KEY='...'

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
assert client.api_key, "OPENAI_API_KEY is not set."


In [ ]:
def build_system_prompt(field_details: str, extra_instructions: str = "", examples: list[dict] | None = None) -> str:
    """Create a strict system prompt that forces filter-only output."""
    examples = examples or []

    example_block = ""
    if examples:
        rendered = []
        for i, ex in enumerate(examples, start=1):
            rendered.append(
                f"Example {i}\nQuestion: {ex['question']}\nFilter: {ex['filter']}"
            )
        example_block = "\n\nFew-shot examples:\n" + "\n\n".join(rendered)

    return dedent(f"""
    You generate ONLY the OData $filter expression from a user question.

    STRICT OUTPUT RULES:
    1) Return only the filter expression text.
    2) Do not include '$filter=' prefix.
    3) Do not include markdown, code fences, explanations, notes, or labels.
    4) Use valid OData filter syntax and field names from the provided field metadata.
    5) If multiple conditions are present, combine with and/or as appropriate.
    6) Use single quotes for string literals and ISO format for dates when needed.

    Field metadata and mapping guidance:
    {field_details}

    Additional instructions:
    {extra_instructions or 'None'}
    {example_block}
    """).strip()


In [ ]:
def generate_odata_filter(
    user_question: str,
    field_details: str,
    extra_instructions: str = "",
    examples: list[dict] | None = None,
    model: str = "gpt-4.1-mini",
) -> str:
    """Call OpenAI and return only the OData filter expression."""
    system_prompt = build_system_prompt(
        field_details=field_details,
        extra_instructions=extra_instructions,
        examples=examples,
    )

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
        temperature=0
    )

    raw = response.output_text.strip()

    # Basic hardening so callers safely append into URL query.
    forbidden_prefixes = ("$filter=", "Filter:", "```")
    if any(raw.startswith(p) for p in forbidden_prefixes):
        # sanitize predictable wrappers
        raw = raw.replace("$filter=", "", 1).replace("Filter:", "", 1).strip().strip('`').strip()

    if "\n" in raw:
        raise ValueError(f"Model output must be a single-line OData filter expression. Got: {raw!r}")

    return raw


In [ ]:
# ---- Configure your domain metadata ----
field_details = dedent("""
Orders entity fields:
- customerName (Edm.String): customer full name
- status (Edm.String): one of 'Open', 'Closed', 'Cancelled'
- orderDate (Edm.DateTimeOffset): order creation timestamp
- totalAmount (Edm.Decimal): total order amount
- region (Edm.String): sales region
""").strip()

extra_instructions = dedent("""
- Interpret 'last month' as current date minus 1 calendar month to now.
- Use ge/le for ranges.
- For status synonyms: 'completed' => Closed.
""").strip()

examples = [
    {
        "question": "Open orders in EMEA over 1000",
        "filter": "status eq 'Open' and region eq 'EMEA' and totalAmount gt 1000",
    },
    {
        "question": "Cancelled orders for Alice",
        "filter": "status eq 'Cancelled' and customerName eq 'Alice'",
    },
]


In [ ]:
# ---- Ask your question ----
user_question = "Show completed orders in EMEA above 5000"
odata_filter = generate_odata_filter(
    user_question=user_question,
    field_details=field_details,
    extra_instructions=extra_instructions,
    examples=examples,
)
odata_filter


In [ ]:
# Example URL assembly (encode in your app as needed)
base_url = "https://example.com/odata/Orders"
query_url = f"{base_url}?$filter={odata_filter}"
query_url
